In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
data_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")



In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import seaborn as sns
df["Delivery_Time"].hist(bins=30, edgecolor='black')

plt.title(f"Target Distribution (Delivery_Time)")
plt.xlabel("Delivery_Time")
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

# 1.	Check if it is skewed (If abs y.skew() > 1.5 and y > 0 then np.log1p(y) and to return it use np.expm1(y))
df["Delivery_Time"].hist()
plt.show()
if abs(df["Delivery_Time"].skew()) > 1.5 and (df["Delivery_Time"] > 0).all():
    df["Delivery_Time"] = np.log1p(df["Delivery_Time"])

In [ ]:
# Task 1: Write your code here:
df.drop("Order_ID" ,axis = 1)

In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)


df["Weather"] = df["Weather"].fillna(df["Weather"].mode()[0])
df["Traffic_Level"] = df["Traffic_Level"].fillna(df["Traffic_Level"].mode()[0])
df["Time_of_Day"] = df["Time_of_Day"].fillna(df["Time_of_Day"].mode()[0])
df["Courier_Experience_yrs"] = df["Courier_Experience_yrs"].fillna(df["Courier_Experience_yrs"].median())



In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)



In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
cat_cols = df.select_dtypes(include=["object"])

encoder = LabelEncoder()

for cat in cat_cols:
  df[cat] = encoder.fit_transform(df[cat])


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df



In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float).dropna()


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

results = {'mae': []}
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


model = RandomForestRegressor(n_estimators=10 ,random_state=42)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

results["mse"].append(mae)


print(f"  MAE:  {np.mean(results['mse']):.4f}")





In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['Lasso'] = model['LASSO Regression'].coef_
coeffs['Ridge'] = model['Ridge Regression'].coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
sns.histplot(data =df , x= df["Delivery_Time"])

In [ ]:
# Task Bonus: Write your code here: